In [3]:
import polars as pl

# Load both sheets
df1 = pl.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df2 = pl.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')

# Combine
df = pl.concat([df1, df2])

print(f"Total rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Total rows: 1,067,371
Columns: 8


In [4]:
df.head(10)

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
i64,str,str,i64,datetime[ms],f64,i64,str
489434,"""85048""","""15CM CHRISTMAS GLASS BALL 20 L…",12,2009-12-01 07:45:00,6.95,13085,"""United Kingdom"""
489434,"""79323P""","""PINK CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,13085,"""United Kingdom"""
489434,"""79323W""",""" WHITE CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,13085,"""United Kingdom"""
489434,"""22041""","""RECORD FRAME 7"" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,13085,"""United Kingdom"""
489434,"""21232""","""STRAWBERRY CERAMIC TRINKET BOX""",24,2009-12-01 07:45:00,1.25,13085,"""United Kingdom"""
489434,"""22064""","""PINK DOUGHNUT TRINKET POT """,24,2009-12-01 07:45:00,1.65,13085,"""United Kingdom"""
489434,"""21871""","""SAVE THE PLANET MUG""",24,2009-12-01 07:45:00,1.25,13085,"""United Kingdom"""
489434,"""21523""","""FANCY FONT HOME SWEET HOME DOO…",10,2009-12-01 07:45:00,5.95,13085,"""United Kingdom"""
489435,"""22350""","""CAT BOWL """,12,2009-12-01 07:46:00,2.55,13085,"""United Kingdom"""


In [5]:
df.null_count()

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
u32,u32,u32,u32,u32,u32,u32,u32
19500,0,4382,0,0,0,243007,0


In [7]:
print(f"Before cleaning: {df.shape[0]:,} rows")

# Step 1: Drop null Customer IDs — can't track retention without identity
df = df.drop_nulls(subset=['Customer ID'])

# Step 2: Drop null Invoices — can't identify the transaction
df = df.drop_nulls(subset=['Invoice'])

# Step 3: Remove cancelled transactions (Invoice starts with 'C')
df = df.filter(~pl.col('Invoice').cast(pl.Utf8).str.starts_with('C'))

# Step 4: Keep only positive Quantity and Price (removes returns and errors)
df = df.filter((pl.col('Quantity') > 0) & (pl.col('Price') > 0))

# Step 5: Add Revenue column
df = df.with_columns(
    (pl.col('Quantity') * pl.col('Price')).alias('Revenue')
)

print(f"After cleaning: {df.shape[0]:,} rows")
print(f"Unique customers: {df['Customer ID'].n_unique():,}")

Before cleaning: 1,067,371 rows
After cleaning: 805,549 rows
Unique customers: 5,878


In [8]:
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


In [15]:
df = df.with_columns(
    pl.col('InvoiceDate').dt.truncate('1mo').alias('InvoiceMonth')
)

monthly = df.group_by('InvoiceMonth').agg(
    pl.col('Invoice').n_unique().alias('transactions'),
    pl.col('Customer ID').n_unique().alias('active_customers'),
    pl.col('Revenue').sum().alias('revenue')
).sort('transactions')

print(monthly)

shape: (25, 4)
┌─────────────────────┬──────────────┬──────────────────┬────────────┐
│ InvoiceMonth        ┆ transactions ┆ active_customers ┆ revenue    │
│ ---                 ┆ ---          ┆ ---              ┆ ---        │
│ datetime[ms]        ┆ u32          ┆ u32              ┆ f64        │
╞═════════════════════╪══════════════╪══════════════════╪════════════╡
│ 2011-12-01 00:00:00 ┆ 778          ┆ 615              ┆ 518210.79  │
│ 2011-01-01 00:00:00 ┆ 987          ┆ 741              ┆ 569445.04  │
│ 2011-02-01 00:00:00 ┆ 997          ┆ 758              ┆ 447137.35  │
│ 2010-01-01 00:00:00 ┆ 1011         ┆ 720              ┆ 557319.062 │
│ 2010-02-01 00:00:00 ┆ 1104         ┆ 772              ┆ 506371.066 │
│ …                   ┆ …            ┆ …                ┆ …          │
│ 2011-09-01 00:00:00 ┆ 1755         ┆ 1266             ┆ 952838.382 │
│ 2011-10-01 00:00:00 ┆ 1929         ┆ 1364             ┆ 1.0393e6   │
│ 2010-10-01 00:00:00 ┆ 2133         ┆ 1497             ┆ 1.03